# 🚢 Learning Machine Learning with the Titanic Dataset

**Goal:** Learn how a machine can *learn patterns from examples* to make predictions.

**The question we will answer:** *Given details about a passenger, can we predict whether they survived the Titanic disaster?*

### The 6 steps we will follow (every ML project uses these!)
1. Look at the data
2. Clean the data
3. Split it into a training set and a test set
4. Train a model
5. Test the model
6. Improve it

> 💡 **How to use this notebook:** Run each code cell with `Shift + Enter`. Read the text before each cell. Try the ✏️ *Your turn* activities.

## What is Machine Learning?

**Traditional programming:** You write the rules. `Data + Rules → Answers`

**Machine learning:** The computer finds the rules by itself from examples. `Data + Answers → Rules`

Here, we give the computer many passengers *along with* whether each one survived. It will discover the rules on its own.

## Step 0: Setup
We load some Python libraries (tools) that other people built for us.

In [ ]:
import pandas as pd                  # tables of data
import numpy as np                   # numbers
import matplotlib.pyplot as plt      # charts
import seaborn as sns                # nicer charts

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

sns.set_theme(style="whitegrid")
print("Setup complete ✅")

## Step 1: Look at the data

In [ ]:
# Load the Titanic dataset (needs internet)
try:
    df = sns.load_dataset("titanic")
except Exception:
    df = pd.read_csv("https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv")

df.columns = df.columns.str.lower()

# Keep only the columns we need
df = df[["survived", "pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]]
df.head()

### What does each column mean?

| Column | Meaning |
|---|---|
| `survived` | **What we want to predict.** 0 = died, 1 = survived |
| `pclass` | Ticket class: 1 = first (expensive), 2 = second, 3 = third (cheap) |
| `sex` | male / female |
| `age` | Age in years |
| `sibsp` | Number of siblings or spouses on board |
| `parch` | Number of parents or children on board |
| `fare` | Ticket price |
| `embarked` | Port where the passenger boarded (S, C, Q) |

In ML language: the columns we use to predict are called **features**. The column we want to predict (`survived`) is called the **label** or **target**.

In [ ]:
print("Rows and columns:", df.shape)
print()
print("How many survived? (0 = died, 1 = survived)")
print(df["survived"].value_counts())

## Step 2: Explore with charts
Before building anything, look for patterns yourself. This is what the machine will also try to find.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.barplot(data=df, x="sex", y="survived", ax=axes[0])
axes[0].set_title("Survival rate by sex")

sns.barplot(data=df, x="pclass", y="survived", ax=axes[1])
axes[1].set_title("Survival rate by ticket class")

sns.histplot(data=df, x="age", hue="survived", bins=20, ax=axes[2])
axes[2].set_title("Age of passengers")

plt.tight_layout()
plt.show()

### 🤔 Discussion
- Who was more likely to survive: men or women? First class or third class?
- What can you say about children?
- Do these patterns match the story "women and children first"?

✏️ **Your turn:** Change `x="sex"` to `x="embarked"` in the first chart and run it. What do you see?

## Step 3: Clean the data

Real data is messy. Computers also cannot work with words like "male" and "female" directly. They need numbers. We need to:
1. Fix **missing values**
2. Convert **words to numbers**

In [ ]:
print("Missing values in each column:")
print(df.isnull().sum())

In [ ]:
# Fill missing ages with the median (middle) age
df["age"] = df["age"].fillna(df["age"].median())

# Fill missing 'embarked' with the most common port
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])

# Convert words to numbers
df["sex"] = df["sex"].map({"male": 0, "female": 1})
df = pd.get_dummies(df, columns=["embarked"], drop_first=False)

print("Missing values left:", df.isnull().sum().sum())
df.head()

✏️ **Your turn:** We filled missing ages with the median. What might go wrong with this? (Hint: are all passengers the same age?)

## Step 4: Split into training and test sets

Imagine a student who memorizes last year's exam answers. They will score 100% on that exam but may fail a new one. We want the model to **understand**, not memorize.

So we hide some passengers from the model:
- **Training set (80%)**: the model learns from these
- **Test set (20%)**: kept secret, used only to check the model at the end

In [ ]:
X = df.drop("survived", axis=1)   # features
y = df["survived"]                # label

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training passengers:", len(X_train))
print("Test passengers:    ", len(X_test))

## Step 5: Train our first model (a Decision Tree)

A decision tree learns a set of yes/no questions, like a game of "20 Questions". It is easy to understand because we can actually *see* the rules it learned.

In [ ]:
tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X_train, y_train)      # <-- this line is the "learning"

print("Model trained ✅")

In [ ]:
plt.figure(figsize=(16, 7))
plot_tree(tree, feature_names=X.columns, class_names=["Died", "Survived"],
          filled=True, rounded=True, fontsize=10)
plt.show()

### How to read the tree
Start at the top box. If the condition is **True**, go left. If **False**, go right. Keep going until you reach the bottom. The colour tells you the prediction (orange = died, blue = survived).

🤔 **Discussion:** What is the very first question the tree asks? Why do you think the machine picked that one?

## Step 6: Test the model

In [ ]:
train_pred = tree.predict(X_train)
test_pred  = tree.predict(X_test)

print(f"Accuracy on training data: {accuracy_score(y_train, train_pred):.1%}")
print(f"Accuracy on test data:     {accuracy_score(y_test,  test_pred):.1%}")

**Accuracy** = the percentage of passengers predicted correctly. 

To see *what kind* of mistakes the model makes, we use a **confusion matrix**.

In [ ]:
cm = confusion_matrix(y_test, test_pred)
ConfusionMatrixDisplay(cm, display_labels=["Died", "Survived"]).plot(cmap="Blues")
plt.title("Confusion matrix (test data)")
plt.show()

### Reading the confusion matrix
- **Top-left:** Died, and the model said "died" ✅
- **Bottom-right:** Survived, and the model said "survived" ✅
- **Top-right:** Died, but the model said "survived" ❌
- **Bottom-left:** Survived, but the model said "died" ❌

## Step 7: Overfitting (memorizing instead of learning)

What happens if we let the tree ask *many* questions? Let's try different depths.

In [ ]:
depths = range(1, 21)
train_scores, test_scores = [], []

for d in depths:
    m = DecisionTreeClassifier(max_depth=d, random_state=42).fit(X_train, y_train)
    train_scores.append(m.score(X_train, y_train))
    test_scores.append(m.score(X_test, y_test))

plt.figure(figsize=(8, 5))
plt.plot(depths, train_scores, label="Training accuracy", marker="o")
plt.plot(depths, test_scores,  label="Test accuracy", marker="o")
plt.xlabel("Tree depth (number of questions allowed)")
plt.ylabel("Accuracy")
plt.title("Overfitting: more complexity is not always better")
plt.legend()
plt.show()

### 🤔 Discussion
- What happens to **training** accuracy as the tree gets deeper?
- What happens to **test** accuracy?
- The gap between them is called **overfitting**. The model memorized the training passengers instead of learning general patterns.

✏️ **Your turn:** Go back to Step 5 and change `max_depth=3` to the depth where test accuracy looked best. Re-run Steps 5 and 6.

## Step 8: Improve it with a Random Forest

One tree can make mistakes. A **random forest** builds many trees and lets them **vote**. Like asking 100 friends instead of one.

In [ ]:
forest = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
forest.fit(X_train, y_train)

print(f"Decision tree test accuracy: {accuracy_score(y_test, tree.predict(X_test)):.1%}")
print(f"Random forest test accuracy: {accuracy_score(y_test, forest.predict(X_test)):.1%}")

### Which features mattered most?

In [ ]:
importance = pd.Series(forest.feature_importances_, index=X.columns).sort_values()
importance.plot(kind="barh", figsize=(8, 5), color="steelblue")
plt.title("Feature importance")
plt.xlabel("How much the model relied on this feature")
plt.show()

🤔 **Discussion:** Which feature mattered most? Does it make sense? What does this tell us about who was given priority during the rescue?

## Step 9: Make your own prediction 🎯
Let's imagine a passenger and ask the model: *would they have survived?*

In [ ]:
# Change these values and re-run!
my_passenger = {
    "pclass": 3,        # 1, 2 or 3
    "sex": 0,           # 0 = male, 1 = female
    "age": 25,
    "sibsp": 0,
    "parch": 0,
    "fare": 8.0,
    "embarked_C": False,
    "embarked_Q": False,
    "embarked_S": True,
}

person = pd.DataFrame([my_passenger])[X.columns]
prob = forest.predict_proba(person)[0][1]

print(f"Chance of survival according to the model: {prob:.0%}")
print("Prediction:", "Survived 🟢" if prob >= 0.5 else "Did not survive 🔴")

✏️ **Your turn:** Try changing the passenger to a 10-year-old girl in first class. Then try a 40-year-old man in third class. Do the results match what you expected?

## Step 10: Think critically ⚠️

Our model is not perfect, and even a perfect model on this dataset would teach us uncomfortable things.

1. **The model learned from history.** It learned that women and first-class passengers survived more. It did not learn what is *fair*, only what *happened*. If a real system learns from biased data, it can repeat the same bias (for example, in loan approvals or hiring).
2. **Small data.** We had fewer than 900 passengers. Real-world AI systems often train on millions of examples.
3. **Accuracy is not everything.** In a hospital, missing a sick patient is far worse than a false alarm. Always ask *what kind* of mistakes the model makes.
4. **Correlation is not causation.** Being in first class did not itself save people. Their cabins were closer to the lifeboats.

## 🎓 Summary: what you learned
| Step | What we did |
|---|---|
| Explore | Looked for patterns in data |
| Clean | Fixed missing values, turned words into numbers |
| Split | Kept a test set hidden to check real learning |
| Train | Let the model find rules from examples |
| Evaluate | Used accuracy and a confusion matrix |
| Improve | Avoided overfitting, tried a random forest |

**The same steps are used** to build spam filters, crop disease detectors, and fraud detection systems. Only the data changes.

## ✏️ Challenge exercises
1. Add a new feature called `family_size = sibsp + parch + 1`. Does accuracy improve?
2. Try `LogisticRegression` from `sklearn.linear_model` instead of a tree. How does it compare?
3. Run Step 4 again with `random_state=7`. Does the accuracy change? Why?